In [293]:

# Install all required libraries
!pip -q install -U yt-dlp
!pip -q install -U openai-whisper
!pip -q install -U google-generativeai
!pip -q install -U pandas
!pip -q install -U ffmpeg-python

# Install FFmpeg
!apt-get -qq update
!apt-get -qq install ffmpeg

print("Libraries Installed Successfully")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Libraries Installed Successfully


In [294]:
import os
import json
import subprocess

import yt_dlp
import whisper
import pandas as pd

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [295]:
folders = [
    "downloads",
    "audio",
    "transcripts",
    "highlights"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created!")

print("\nFolder Structure:")
for folder in folders:
    print("📁", folder)

Project folders created!

Folder Structure:
📁 downloads
📁 audio
📁 transcripts
📁 highlights


In [296]:
youtube_url = input("🎥 Enter YouTube URL: ").strip()

print("\nYou entered:")
print(youtube_url)

🎥 Enter YouTube URL: https://www.youtube.com/watch?v=eVFzbxmKNUw

You entered:
https://www.youtube.com/watch?v=eVFzbxmKNUw


In [297]:
import yt_dlp
import os

def download_video(youtube_url):

    print("\nStarting download...\n")

    ydl_opts = {
        "format": "bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "outtmpl": "downloads/video.%(ext)s",
        "noplaylist": True,
        "quiet": False,
        "geo_bypass": True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([youtube_url])

        print("\n Download completed!")

    except Exception as e:
        print("\n Download failed!")
        print(e)

In [298]:
download_video(youtube_url)


Starting download...

[youtube] Extracting URL: https://www.youtube.com/watch?v=eVFzbxmKNUw
[youtube] eVFzbxmKNUw: Downloading webpage


[youtube] eVFzbxmKNUw: Downloading android vr player API JSON
[info] eVFzbxmKNUw: Downloading 1 format(s): 401+251
[download] downloads/video.mp4 has already been downloaded

 Download completed!


In [299]:
import os

video_path = None

for file in os.listdir("downloads"):
    if file.endswith((".mp4", ".mkv", ".webm", ".mov")):
        video_path = os.path.join("downloads", file)
        break

if video_path:
    print("Video Found:")
    print(video_path)
else:
    print("No video found!")

Video Found:
downloads/video.mp4


In [300]:
audio_path = "audio/audio.wav"

command = [
    "ffmpeg",
    "-i",
    video_path,
    "-ar",
    "16000",
    "-ac",
    "1",
    audio_path,
    "-y"
]

subprocess.run(command)

print("Audio Extracted Successfully!")
print(audio_path)

Audio Extracted Successfully!
audio/audio.wav


In [301]:
import os

if os.path.exists(audio_path):
    print("Audio file created successfully!")
else:
    print("Audio extraction failed.")

Audio file created successfully!


In [302]:
import whisper

print("Loading Whisper model...")

# Choose one:
# tiny   -> Fastest, least accurate
# base   -> Good for testing
# small  -> Better accuracy
# medium -> Even better (requires more GPU memory)

model = whisper.load_model("base")

print("Whisper model loaded successfully!")

Loading Whisper model...
Whisper model loaded successfully!


In [303]:
print("Transcribing audio...")

result = model.transcribe(
    audio_path,
    language="en"
)

segments = result["segments"]

print(f"Transcription completed!")
print(f"Total segments: {len(segments)}")

  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Transcribing audio...
Transcription completed!
Total segments: 151


In [304]:

import pandas as pd

transcript = []

for seg in segments:
    transcript.append({
        "Start": seg["start"],
        "End": seg["end"],
        "Text": seg["text"].strip()
    })

df = pd.DataFrame(transcript)

df.head(10)

,Start,End,Text
0,0.00,10.72,So I'm sure everyone here has been on the givi...
1,10.72,11.72,failure.
2,11.72,12.72,Is that fair?
3,12.72,17.58,"Well, I want to share a time when I sent a mes..."
4,17.58,23.24,"hours of wasted effort, a few sleepless nights..."
5,23.24,26.08,That message was sure.
6,26.08,27.92,"Yep, that's it."
7,27.92,34.04,"One word, four letters, no punctuation, no emo..."
8,34.04,37.72,I bet you were expecting something a little bi...
9,37.72,39.32,So here's what happens.


In [305]:
import pandas as pd

rows = []

for seg in segments:

    rows.append({

        "start": float(seg["start"]),
        "end": float(seg["end"]),
        "duration": round(seg["end"]-seg["start"],2),
        "text": seg["text"]

    })

df = pd.DataFrame(rows)

print("Total Segments :",len(df))

df.head()

Total Segments : 151


,start,end,duration,text
0,0.00,10.72,10.72,So I'm sure everyone here has been on the giv...
1,10.72,11.72,1.00,failure.
2,11.72,12.72,1.00,Is that fair?
3,12.72,17.58,4.86,"Well, I want to share a time when I sent a me..."
4,17.58,23.24,5.66,"hours of wasted effort, a few sleepless night..."


In [306]:
import re

def clean_text(text):

    text = str(text)

    text = text.replace("\n"," ")

    text = re.sub(r"\s+"," ",text)

    text = text.strip()

    return text

df["clean_text"] = df["text"].apply(clean_text)

df.head()

,start,end,duration,text,clean_text
0,0.00,10.72,10.72,So I'm sure everyone here has been on the giv...,So I'm sure everyone here has been on the givi...
1,10.72,11.72,1.00,failure.,failure.
2,11.72,12.72,1.00,Is that fair?,Is that fair?
3,12.72,17.58,4.86,"Well, I want to share a time when I sent a me...","Well, I want to share a time when I sent a mes..."
4,17.58,23.24,5.66,"hours of wasted effort, a few sleepless night...","hours of wasted effort, a few sleepless nights..."


In [307]:

before = len(df)

df = df[df["clean_text"]!=""]

after = len(df)

print("Removed :",before-after)
print("Remaining :",after)

Removed : 0
Remaining : 151


In [308]:
df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))

df = df[df["word_count"]>=3]

print("Remaining Segments :",len(df))

Remaining Segments : 142


In [309]:
before = len(df)

df = df.drop_duplicates(subset="clean_text")

after = len(df)

print("Duplicates Removed :",before-after)

Duplicates Removed : 0


In [310]:
df = df.reset_index(drop=True)

df.head()

,start,end,duration,text,clean_text,word_count
0,0.00,10.72,10.72,So I'm sure everyone here has been on the giv...,So I'm sure everyone here has been on the givi...,17
1,11.72,12.72,1.00,Is that fair?,Is that fair?,3
2,12.72,17.58,4.86,"Well, I want to share a time when I sent a me...","Well, I want to share a time when I sent a mes...",18
3,17.58,23.24,5.66,"hours of wasted effort, a few sleepless night...","hours of wasted effort, a few sleepless nights...",13
4,23.24,26.08,2.84,That message was sure.,That message was sure.,4


In [311]:
merged = []

i = 0

while i < len(df):

    current = df.iloc[i].copy()

    while (

        i+1 < len(df)

        and current["word_count"]<8

    ):

        nxt = df.iloc[i+1]

        current["clean_text"] += " " + nxt["clean_text"]

        current["end"] = nxt["end"]

        current["duration"] = round(

            current["end"]-current["start"],2

        )

        current["word_count"] = len(

            current["clean_text"].split()

        )

        i += 1

    merged.append(current)

    i += 1

merged_df = pd.DataFrame(merged)

print("Merged Segments :",len(merged_df))

Merged Segments : 104


In [312]:
def timestamp(sec):

    m = int(sec//60)

    s = int(sec%60)

    return f"{m:02d}:{s:02d}"

merged_df["timestamp"] = merged_df["start"].apply(timestamp)

merged_df.head()

,start,end,duration,text,clean_text,word_count,timestamp
0,0.00,10.72,10.72,So I'm sure everyone here has been on the giv...,So I'm sure everyone here has been on the givi...,17,00:00
1,11.72,17.58,5.86,Is that fair?,"Is that fair? Well, I want to share a time whe...",21,00:11
3,17.58,23.24,5.66,"hours of wasted effort, a few sleepless night...","hours of wasted effort, a few sleepless nights...",13,00:17
4,23.24,34.04,10.80,That message was sure.,"That message was sure. Yep, that's it. One wor...",17,00:23
7,34.04,37.72,3.68,I bet you were expecting something a little b...,I bet you were expecting something a little bi...,13,00:34


In [313]:
merged_df.to_csv(

    "transcripts/clean_transcript.csv",

    index=False

)

print("Saved Successfully")

Saved Successfully


In [314]:
merged_df[
    [
        "timestamp",
        "clean_text",
        "word_count"
    ]
].head(20)

,timestamp,clean_text,word_count
0,00:00,So I'm sure everyone here has been on the givi...,17
1,00:11,"Is that fair? Well, I want to share a time whe...",21
3,00:17,"hours of wasted effort, a few sleepless nights...",13
4,00:23,"That message was sure. Yep, that's it. One wor...",17
7,00:34,I bet you were expecting something a little bi...,13
8,00:37,So here's what happens. My team was working on...,19
10,00:43,"on deadline, the project manager Charlie sent ...",15
11,00:47,are we okay to send this to the board chair fo...,12
12,00:51,I reviewed the presentation that evening while...,13
13,00:56,"It was spot on, key messages, graphics, so muc...",17


In [315]:
conversation_data = []

for i in range(len(merged_df)):

    previous_text = ""
    current_text = merged_df.iloc[i]["clean_text"]
    next_text = ""

    if i > 0:
        previous_text = merged_df.iloc[i-1]["clean_text"]

    if i < len(merged_df)-1:
        next_text = merged_df.iloc[i+1]["clean_text"]

    conversation = f"""
Previous:
{previous_text}

Current:
{current_text}

Next:
{next_text}
"""

    conversation_data.append({
        "start": merged_df.iloc[i]["start"],
        "end": merged_df.iloc[i]["end"],
        "timestamp": merged_df.iloc[i]["timestamp"],
        "conversation": conversation
    })

conversation_df = pd.DataFrame(conversation_data)

print("Conversation windows created:", len(conversation_df))

Conversation windows created: 104


In [316]:
conversation_df.head(5)

,start,end,timestamp,conversation
0,0.00,10.72,00:00,\nPrevious:\n\n\nCurrent:\nSo I'm sure everyon...
1,11.72,17.58,00:11,\nPrevious:\nSo I'm sure everyone here has bee...
2,17.58,23.24,00:17,"\nPrevious:\nIs that fair? Well, I want to sha..."
3,23.24,34.04,00:23,"\nPrevious:\nhours of wasted effort, a few sle..."
4,34.04,37.72,00:34,"\nPrevious:\nThat message was sure. Yep, that'..."


In [317]:
print(conversation_df.loc[5, "conversation"])


Previous:
I bet you were expecting something a little bit more controversial or spicy.

Current:
So here's what happens. My team was working on an important presentation for an upcoming board meeting, and right

Next:
on deadline, the project manager Charlie sent me an email with a presentation and said,



In [318]:
def count_words(text):
    return len(text.split())

def question_mark(text):
    return int("?" in text)

conversation_df["word_count"] = merged_df["clean_text"].apply(count_words)

conversation_df["contains_question"] = merged_df["clean_text"].apply(question_mark)

conversation_df["duration"] = merged_df["duration"]

conversation_df.head()

,start,end,timestamp,conversation,word_count,contains_question,duration
0,0.00,10.72,00:00,\nPrevious:\n\n\nCurrent:\nSo I'm sure everyon...,17.0,0.0,10.72
1,11.72,17.58,00:11,\nPrevious:\nSo I'm sure everyone here has bee...,21.0,1.0,5.86
2,17.58,23.24,00:17,"\nPrevious:\nIs that fair? Well, I want to sha...",NaN,NaN,NaN
3,23.24,34.04,00:23,"\nPrevious:\nhours of wasted effort, a few sle...",13.0,0.0,5.66
4,34.04,37.72,00:34,"\nPrevious:\nThat message was sure. Yep, that'...",17.0,0.0,10.80


In [319]:
agreement_words = [
    "yes",
    "yeah",
    "exactly",
    "correct",
    "right",
    "absolutely",
    "true",
    "definitely",
    "agree"
]

def agreement_score(text):

    text = text.lower()

    score = 0

    for word in agreement_words:
        if word in text:
            score += 1

    return score

conversation_df["agreement_score"] = merged_df["clean_text"].apply(agreement_score)

In [320]:
disagreement_words = [
    "no",
    "not",
    "never",
    "don't",
    "disagree",
    "wrong",
    "however",
    "actually",
    "but"
]

def disagreement_score(text):

    text = text.lower()

    score = 0

    for word in disagreement_words:
        if word in text:
            score += 1

    return score

conversation_df["disagreement_score"] = merged_df["clean_text"].apply(disagreement_score)

In [321]:
question_words = [
    "what",
    "why",
    "when",
    "where",
    "who",
    "which",
    "how",
    "can",
    "could",
    "should",
    "would",
    "do",
    "does",
    "did"
]

def question_score(text):

    text = text.lower()

    score = 0

    for word in question_words:
        if word in text:
            score += 1

    if "?" in text:
        score += 2

    return score

conversation_df["question_score"] = merged_df["clean_text"].apply(question_score)

In [322]:
conversation_df[
    [
        "timestamp",
        "word_count",
        "contains_question",
        "question_score",
        "agreement_score",
        "disagreement_score"
    ]
].head(20)

,timestamp,word_count,contains_question,question_score,agreement_score,disagreement_score
0,00:00,17.0,0.0,0.0,0.0,0.0
1,00:11,21.0,1.0,3.0,0.0,0.0
2,00:17,NaN,NaN,NaN,NaN,NaN
3,00:23,13.0,0.0,0.0,0.0,0.0
4,00:34,17.0,0.0,0.0,0.0,1.0
5,00:37,NaN,NaN,NaN,NaN,NaN
6,00:43,NaN,NaN,NaN,NaN,NaN
7,00:47,13.0,0.0,0.0,0.0,0.0
8,00:51,19.0,0.0,1.0,1.0,0.0
9,00:56,NaN,NaN,NaN,NaN,NaN


In [323]:
conversation_df.to_csv(
    "transcripts/conversation_features.csv",
    index=False
)

print("Feature dataset saved!")

Feature dataset saved!


In [324]:
!pip -q install scikit-learn

In [325]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

print("TF-IDF Ready")

TF-IDF Ready


In [326]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=30,
    ngram_range=(1,2)
)

tfidf_matrix = vectorizer.fit_transform(
    conversation_df["conversation"]
)

feature_names = vectorizer.get_feature_names_out()

print("Top Features Extracted:", len(feature_names))

Top Features Extracted: 30


In [327]:
print("Top Keywords:\n")

for word in feature_names:
    print(word)

Top Keywords:

best
best communicate
better
communication
communication style
current
day
email
going
ideas
information
just
kate
like
maybe
meeting
messages
mike
presentation
previous
said
share
style
style tag
sure
tag
team
text
think
time


In [328]:

tfidf_scores = np.asarray(tfidf_matrix.sum(axis=1)).flatten()

conversation_df["tfidf_score"] = tfidf_scores

conversation_df[
    [
        "timestamp",
        "tfidf_score"
    ]
].head()

,timestamp,tfidf_score
0,00:00,2.453802
1,00:11,2.643024
2,00:17,2.421259
3,00:23,1.784209
4,00:34,2.217211


In [329]:

tfidf_scores = np.asarray(tfidf_matrix.sum(axis=1)).flatten()

conversation_df["tfidf_score"] = tfidf_scores

conversation_df[
    [
        "timestamp",
        "tfidf_score"
    ]
].head()

,timestamp,tfidf_score
0,00:00,2.453802
1,00:11,2.643024
2,00:17,2.421259
3,00:23,1.784209
4,00:34,2.217211


In [330]:
conversation_df[
    [
        "timestamp",
        "tfidf_score",
        "conversation"
    ]
].head(10)

,timestamp,tfidf_score,conversation
0,00:00,2.453802,\nPrevious:\n\n\nCurrent:\nSo I'm sure everyon...
1,00:11,2.643024,\nPrevious:\nSo I'm sure everyone here has bee...
2,00:17,2.421259,"\nPrevious:\nIs that fair? Well, I want to sha..."
3,00:23,1.784209,"\nPrevious:\nhours of wasted effort, a few sle..."
4,00:34,2.217211,"\nPrevious:\nThat message was sure. Yep, that'..."
5,00:37,2.204716,\nPrevious:\nI bet you were expecting somethin...
6,00:43,2.204716,\nPrevious:\nSo here's what happens. My team w...
7,00:47,1.773788,"\nPrevious:\non deadline, the project manager ..."
8,00:51,1.768665,\nPrevious:\nare we okay to send this to the b...
9,00:56,1.768665,\nPrevious:\nI reviewed the presentation that ...


In [331]:
!pip -q install vaderSentiment

In [332]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return abs(analyzer.polarity_scores(text)["compound"])

conversation_df["sentiment_score"] = conversation_df["conversation"].apply(get_sentiment)

conversation_df[["timestamp","sentiment_score"]].head()

,timestamp,sentiment_score
0,00:00,0.8974
1,00:11,0.7769
2,00:17,0.8068
3,00:23,0.1132
4,00:34,0.6661


In [333]:
features = [

    "question_score",
    "agreement_score",
    "disagreement_score",
    "word_count",
    "duration",
    "sentiment_score",
    "tfidf_score"

]

for feature in features:

    max_val = conversation_df[feature].max()

    if max_val > 0:

        conversation_df[feature] = conversation_df[feature] / max_val

print("Normalization Complete")

Normalization Complete


In [334]:

WEIGHTS = {

    "question" : 0.25,
    "agreement" : 0.15,
    "disagreement" : 0.15,
    "tfidf" : 0.20,
    "sentiment" : 0.15,
    "word" : 0.05,
    "duration" : 0.05

}

In [335]:
conversation_df["highlight_score"] = (

WEIGHTS["question"] * conversation_df["question_score"]

+

WEIGHTS["agreement"] * conversation_df["agreement_score"]

+

WEIGHTS["disagreement"] * conversation_df["disagreement_score"]

+

WEIGHTS["tfidf"] * conversation_df["tfidf_score"]

+

WEIGHTS["sentiment"] * conversation_df["sentiment_score"]

+

WEIGHTS["word"] * conversation_df["word_count"]

+

WEIGHTS["duration"] * conversation_df["duration"]

)

In [336]:
conversation_df["highlight_score"] = (

conversation_df["highlight_score"] * 100

).round(2)

In [337]:
def classify(row):

    if row["question_score"] >= 0.40:
        return "Question"

    elif row["agreement_score"] >= row["disagreement_score"] and row["agreement_score"] >= 0.30:
        return "Agreement"

    elif row["disagreement_score"] >= 0.30:
        return "Disagreement"

    else:
        return "Other"

conversation_df["event"] = conversation_df.apply(classify, axis=1)

In [338]:

ranked_df = conversation_df.sort_values(

    by="highlight_score",

    ascending=False

).reset_index(drop=True)

ranked_df["rank"] = ranked_df.index + 1

ranked_df.head(20)


,start,end,timestamp,conversation,word_count,contains_question,duration,agreement_score,disagreement_score,question_score,tfidf_score,sentiment_score,highlight_score,event,rank
0,159.96,167.44,02:39,\nPrevious:\nSo I want to provide a simple sol...,0.80,1.0,0.738095,1.0,1.000000,0.50,0.695553,0.971215,78.67,Question,1
1,471.28,478.64,07:51,"\nPrevious:\nText communication style tag, ema...",0.48,1.0,0.270833,0.0,0.666667,1.00,0.864994,0.889016,69.39,Question,2
2,91.92,100.92,01:31,\nPrevious:\nI called him up on the phone and ...,0.92,1.0,0.413690,0.0,1.000000,0.75,0.609504,0.375351,58.24,Question,3
3,466.24,471.28,07:46,\nPrevious:\nHe lived in detailed spreadsheets...,0.48,1.0,0.291667,0.0,0.333333,0.75,0.779974,0.815234,55.44,Question,4
4,220.48,225.16,03:40,"\nPrevious:\nOne, how we best communicate, ema...",0.72,1.0,0.666667,0.0,0.000000,0.75,0.759348,0.947522,55.08,Question,5
5,11.72,17.58,00:11,\nPrevious:\nSo I'm sure everyone here has bee...,0.84,1.0,0.436012,0.0,0.000000,0.75,0.791744,0.807337,53.07,Question,6
6,362.76,368.56,06:02,\nPrevious:\nMike would never respond to the m...,0.88,0.0,0.696429,1.0,0.000000,0.25,0.664719,0.685545,52.71,Agreement,7
7,212.92,220.48,03:32,\nPrevious:\nstyle tag that covers two critica...,0.72,0.0,0.556548,0.0,0.000000,0.50,0.996130,0.839031,51.39,Question,8
8,324.56,329.88,05:24,\nPrevious:\nstyles and share your own as well...,0.80,0.0,0.413690,0.0,0.666667,0.25,0.850282,0.791333,51.19,Disagreement,9
9,244.52,252.40,04:04,\nPrevious:\ncontacts and advanced for fastest...,0.76,0.0,0.380952,0.0,0.333333,0.50,0.681212,0.951990,51.11,Question,10


In [339]:
import os

os.makedirs("output", exist_ok=True)

ranked_df.to_csv(

    "WEEK1_HIGHLIGHT EXTRACTION.csv",

    index=False

)

print("Saved Successfully!")

Saved Successfully!


In [340]:
ranked_df[

[
    "rank",
    "timestamp",
    "event",
    "highlight_score",
    "conversation"

]

].head(20)

,rank,timestamp,event,highlight_score,conversation
0,1,02:39,Question,78.67,\nPrevious:\nSo I want to provide a simple sol...
1,2,07:51,Question,69.39,"\nPrevious:\nText communication style tag, ema..."
2,3,01:31,Question,58.24,\nPrevious:\nI called him up on the phone and ...
3,4,07:46,Question,55.44,\nPrevious:\nHe lived in detailed spreadsheets...
4,5,03:40,Question,55.08,"\nPrevious:\nOne, how we best communicate, ema..."
5,6,00:11,Question,53.07,\nPrevious:\nSo I'm sure everyone here has bee...
6,7,06:02,Agreement,52.71,\nPrevious:\nMike would never respond to the m...
7,8,03:32,Question,51.39,\nPrevious:\nstyle tag that covers two critica...
8,9,05:24,Disagreement,51.19,\nPrevious:\nstyles and share your own as well...
9,10,04:04,Question,51.11,\nPrevious:\ncontacts and advanced for fastest...


In [341]:
from google.colab import files

files.download("WEEK1_HIGHLIGHT EXTRACTION.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [342]:
# Keep only one highlight within a 20-second window
MIN_GAP = 20

selected = []

for _, row in ranked_df.iterrows():

    current_time = row["start"]

    keep = True

    for previous in selected:

        if abs(current_time - previous["start"]) < MIN_GAP:

            keep = False
            break

    if keep:
        selected.append(row)

In [343]:
# Create dataframe after removing overlaps
selected_df = pd.DataFrame(selected)

selected_df = selected_df.sort_values(
    "highlight_score",
    ascending=False
)

selected_df = selected_df.reset_index(
    drop=True
)

print("Original highlights:")
print(len(ranked_df))

print("\nFiltered highlights:")
print(len(selected_df))

selected_df.head(20)

Original highlights:
104

Filtered highlights:
21


,start,end,timestamp,conversation,word_count,contains_question,duration,agreement_score,disagreement_score,question_score,tfidf_score,sentiment_score,highlight_score,event,rank
0,159.96,167.44,02:39,\nPrevious:\nSo I want to provide a simple sol...,0.80,1.0,0.738095,1.0,1.000000,0.50,0.695553,0.971215,78.67,Question,1
1,471.28,478.64,07:51,"\nPrevious:\nText communication style tag, ema...",0.48,1.0,0.270833,0.0,0.666667,1.00,0.864994,0.889016,69.39,Question,2
2,91.92,100.92,01:31,\nPrevious:\nI called him up on the phone and ...,0.92,1.0,0.413690,0.0,1.000000,0.75,0.609504,0.375351,58.24,Question,3
3,220.48,225.16,03:40,"\nPrevious:\nOne, how we best communicate, ema...",0.72,1.0,0.666667,0.0,0.000000,0.75,0.759348,0.947522,55.08,Question,5
4,11.72,17.58,00:11,\nPrevious:\nSo I'm sure everyone here has bee...,0.84,1.0,0.436012,0.0,0.000000,0.75,0.791744,0.807337,53.07,Question,6
5,362.76,368.56,06:02,\nPrevious:\nMike would never respond to the m...,0.88,0.0,0.696429,1.0,0.000000,0.25,0.664719,0.685545,52.71,Agreement,7
6,324.56,329.88,05:24,\nPrevious:\nstyles and share your own as well...,0.80,0.0,0.413690,0.0,0.666667,0.25,0.850282,0.791333,51.19,Disagreement,9
7,244.52,252.40,04:04,\nPrevious:\ncontacts and advanced for fastest...,0.76,0.0,0.380952,0.0,0.333333,0.50,0.681212,0.951990,51.11,Question,10
8,567.40,574.48,09:27,\nPrevious:\nfor our relationship too. And thi...,0.56,1.0,0.434524,0.0,0.000000,0.75,0.664967,0.885483,50.30,Question,11
9,419.74,421.76,06:59,\nPrevious:\nDo you do your best thinking in t...,0.60,0.0,0.395833,0.0,0.666667,0.25,0.624617,0.995635,48.66,Disagreement,13


In [344]:
print(ranked_df.columns)

Index(['start', 'end', 'timestamp', 'conversation', 'word_count',
       'contains_question', 'duration', 'agreement_score',
       'disagreement_score', 'question_score', 'tfidf_score',
       'sentiment_score', 'highlight_score', 'event', 'rank'],
      dtype='str')


In [345]:
# Remove overlapping highlights

MIN_GAP = 20  # Minimum gap between highlights in seconds

selected = []

for _, row in ranked_df.iterrows():

    current_time = row["start"]

    keep = True

    for previous in selected:

        # Ignore highlights that are too close together
        if abs(current_time - previous["start"]) < MIN_GAP:
            keep = False
            break

    if keep:
        selected.append(row)

In [346]:
# Create dataframe after removing overlaps

selected_df = pd.DataFrame(selected)

selected_df = selected_df.sort_values(
    "highlight_score",
    ascending=False
)

selected_df = selected_df.reset_index(
    drop=True
)

print("Original highlights:", len(ranked_df))
print("Filtered highlights:", len(selected_df))

selected_df.head(10)

Original highlights: 104
Filtered highlights: 21


,start,end,timestamp,conversation,word_count,contains_question,duration,agreement_score,disagreement_score,question_score,tfidf_score,sentiment_score,highlight_score,event,rank
0,159.96,167.44,02:39,\nPrevious:\nSo I want to provide a simple sol...,0.80,1.0,0.738095,1.0,1.000000,0.50,0.695553,0.971215,78.67,Question,1
1,471.28,478.64,07:51,"\nPrevious:\nText communication style tag, ema...",0.48,1.0,0.270833,0.0,0.666667,1.00,0.864994,0.889016,69.39,Question,2
2,91.92,100.92,01:31,\nPrevious:\nI called him up on the phone and ...,0.92,1.0,0.413690,0.0,1.000000,0.75,0.609504,0.375351,58.24,Question,3
3,220.48,225.16,03:40,"\nPrevious:\nOne, how we best communicate, ema...",0.72,1.0,0.666667,0.0,0.000000,0.75,0.759348,0.947522,55.08,Question,5
4,11.72,17.58,00:11,\nPrevious:\nSo I'm sure everyone here has bee...,0.84,1.0,0.436012,0.0,0.000000,0.75,0.791744,0.807337,53.07,Question,6
5,362.76,368.56,06:02,\nPrevious:\nMike would never respond to the m...,0.88,0.0,0.696429,1.0,0.000000,0.25,0.664719,0.685545,52.71,Agreement,7
6,324.56,329.88,05:24,\nPrevious:\nstyles and share your own as well...,0.80,0.0,0.413690,0.0,0.666667,0.25,0.850282,0.791333,51.19,Disagreement,9
7,244.52,252.40,04:04,\nPrevious:\ncontacts and advanced for fastest...,0.76,0.0,0.380952,0.0,0.333333,0.50,0.681212,0.951990,51.11,Question,10
8,567.40,574.48,09:27,\nPrevious:\nfor our relationship too. And thi...,0.56,1.0,0.434524,0.0,0.000000,0.75,0.664967,0.885483,50.30,Question,11
9,419.74,421.76,06:59,\nPrevious:\nDo you do your best thinking in t...,0.60,0.0,0.395833,0.0,0.666667,0.25,0.624617,0.995635,48.66,Disagreement,13


In [347]:
selected_df[
    ["start", "event", "highlight_score"]
].head(10)

,start,event,highlight_score
0,159.96,Question,78.67
1,471.28,Question,69.39
2,91.92,Question,58.24
3,220.48,Question,55.08
4,11.72,Question,53.07
5,362.76,Agreement,52.71
6,324.56,Disagreement,51.19
7,244.52,Question,51.11
8,567.40,Question,50.30
9,419.74,Disagreement,48.66
